# Revision R5 — Peripheral gradient, object extent, and class specificity

This notebook addresses the five requirements in editorial comment R5:

1. estimate a simple image-level object-extent covariate;
2. control for object extent using partial rank correlations and pooled class-interaction models;
3. repeat the analysis for all six TrashNet classes;
4. test the claim that cardboard has the highest median peripheral-gradient removal;
5. report results for both angular sensitivity and relative Euclidean distance.

The notebook does **not** recompute VGG-16 embeddings and does **not** require a GPU. It reads the original TrashNet images and the existing complete metrics CSV.

In [ ]:
# 1. Mount Google Drive and define the main paths.
from google.colab import drive
drive.mount('/content/gdrive')

from pathlib import Path

DRIVE_ROOT = Path('/content/gdrive/MyDrive')

# This path was recovered from the original experiment manifest.
DATASET_ROOT = DRIVE_ROOT / '2026-0' / 'DatasetOriginalCompleto'

# Leave as None to search for the exact metrics filename in MyDrive.
# If more than one candidate is found, replace None with the correct full path.
METRICS_CSV = None

OUTPUT_ROOT = DRIVE_ROOT / 'Revision_R5_Peripheral_Gradient_VGG16'

SEED = 2026
N_BOOT_MEDIAN = 2000
N_BOOT_PARTIAL = 1000

print('Dataset root:', DATASET_ROOT)
print('Output root:', OUTPUT_ROOT)

In [ ]:
# 2. Imports, software versions, and deterministic configuration.
import json
import hashlib
import math
import os
import platform
import shutil
import subprocess
import sys
import warnings
import zipfile

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageDraw
import scipy
from scipy.stats import (chi2 as chi2_dist, kruskal, mannwhitneyu,
                         pearsonr, rankdata, spearmanr, t as t_dist)
import seaborn as sns
try:
    import statsmodels
    import statsmodels.formula.api as smf
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    import statsmodels
    import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

rng = np.random.default_rng(SEED)
np.random.seed(SEED)

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TABLES = OUTPUT_ROOT / 'tables'
FIGURES = OUTPUT_ROOT / 'figures'
QC = OUTPUT_ROOT / 'quality_control'
for folder in [TABLES, FIGURES, QC]:
    folder.mkdir(parents=True, exist_ok=True)

VERSIONS = {
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scipy': scipy.__version__,
    'opencv': cv2.__version__,
    'Pillow': PIL.__version__,
    'matplotlib': matplotlib.__version__,
    'seaborn': sns.__version__,
    'statsmodels': statsmodels.__version__,
    'random_seed': SEED,
    'bootstrap_replicates_median': N_BOOT_MEDIAN,
    'bootstrap_replicates_partial_correlation': N_BOOT_PARTIAL,
}
print(json.dumps(VERSIONS, indent=2))

## What “object extent” means here

TrashNet objects are commonly photographed against a relatively plain background. The notebook estimates the background colour from the image border, separates pixels that differ from that background using Otsu's threshold, cleans the mask, and calculates:

- **bounding-box fraction**: area of the foreground bounding box divided by total image area — the primary covariate requested by the editor;
- **foreground fraction**: number of foreground pixels divided by total image area — a secondary sensitivity check.

This is a simple reproducible proxy, not a semantic segmentation model. A montage is generated for visual quality control.

In [ ]:
# 3. Resolve the metrics CSV and validate the original image collection.
EXPECTED_COUNTS = {
    'cardboard': 403, 'glass': 501, 'metal': 410,
    'paper': 594, 'plastic': 482, 'trash': 137,
}
CLASSES = list(EXPECTED_COUNTS)
LEVELS = ['low', 'medium', 'high']
ZOOM_FACTORS = {'low': 1.10, 'medium': 1.25, 'high': 1.50}
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

if METRICS_CSV is None:
    candidates = sorted(DRIVE_ROOT.rglob('embeddings_metricas_VGG16_TrashNet_completo_v2*.csv'))
    print('Metrics candidates found:')
    for candidate in candidates:
        print(' -', candidate)
    if len(candidates) > 1:
        hashes = {}
        for candidate in candidates:
            digest = hashlib.sha256(candidate.read_bytes()).hexdigest()
            hashes.setdefault(digest, []).append(candidate)
        if len(hashes) == 1:
            METRICS_CSV = candidates[0]
            print('All candidates are byte-identical; using:', METRICS_CSV)
        else:
            raise RuntimeError(
                'Several non-identical metrics CSV files were found. '
                'Set METRICS_CSV manually in Cell 1 and run again.'
            )
    elif len(candidates) == 1:
        METRICS_CSV = candidates[0]
    else:
        raise RuntimeError(
            'The notebook did not find the metrics CSV. '
            'Set METRICS_CSV manually in Cell 1 and run again.'
        )

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f'Original-image folder not found: {DATASET_ROOT}. '
        'Update DATASET_ROOT in Cell 1.'
    )

image_rows = []
for class_name, expected_n in EXPECTED_COUNTS.items():
    class_dir = DATASET_ROOT / class_name
    if not class_dir.exists():
        raise FileNotFoundError(f'Missing class folder: {class_dir}')
    files = sorted(p for p in class_dir.iterdir() if p.suffix.lower() in VALID_EXTENSIONS)
    if len(files) != expected_n:
        raise RuntimeError(f'{class_name}: expected {expected_n} images, found {len(files)}')
    image_rows.extend({
        'class': class_name,
        'original_filename': p.name,
        'image_id': p.stem,
        'original_path': str(p),
    } for p in files)

images = pd.DataFrame(image_rows)
assert len(images) == 2527
assert not images.duplicated(['class', 'original_filename']).any()

metrics = pd.read_csv(METRICS_CSV)
required = {
    'image_id', 'original_filename', 'class', 'transformation', 'level',
    'block', 'cosine_similarity', 'relative_euclidean_distance'
}
missing = required - set(metrics.columns)
if missing:
    raise RuntimeError(f'Missing required metrics columns: {sorted(missing)}')

zoom_metrics = metrics[
    (metrics['transformation'] == 'zoom') & (metrics['block'] == 'Block5')
].copy()
assert len(zoom_metrics) == 2527 * 3
assert not zoom_metrics.duplicated(['image_id', 'level']).any()

print('Original images:', len(images))
print('Zoom/Block5 metric rows:', len(zoom_metrics))
print('Metrics file:', METRICS_CSV)
display(images.groupby('class').size().rename('n').to_frame())

In [ ]:
# 4. Image-processing functions: foreground extent and peripheral-gradient removal.
def clean_foreground_mask(rgb):
    # Return a simple foreground mask and reproducibility diagnostics.
    h, w = rgb.shape[:2]
    border_width = max(5, int(round(min(h, w) * 0.05)))
    border = np.concatenate([
        rgb[:border_width].reshape(-1, 3),
        rgb[-border_width:].reshape(-1, 3),
        rgb[:, :border_width].reshape(-1, 3),
        rgb[:, -border_width:].reshape(-1, 3),
    ], axis=0)
    background_rgb = np.median(border, axis=0)

    distance = np.linalg.norm(rgb.astype(np.float32) - background_rgb, axis=2)
    max_rgb_distance = math.sqrt(3 * 255 ** 2)
    distance_u8 = np.clip(distance / max_rgb_distance * 255, 0, 255).astype(np.uint8)
    otsu_threshold, mask = cv2.threshold(
        distance_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    kernel_size = max(3, int(round(min(h, w) * 0.01)))
    if kernel_size % 2 == 0:
        kernel_size += 1
    kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    min_component_area = max(25, int(round(h * w * 0.001)))
    keep = []
    if n_labels > 1:
        keep = [label for label in range(1, n_labels)
                if stats[label, cv2.CC_STAT_AREA] >= min_component_area]
        if not keep:
            keep = [1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))]

    cleaned = np.isin(labels, keep).astype(np.uint8) if keep else np.zeros((h, w), np.uint8)
    ys, xs = np.where(cleaned > 0)
    if len(xs) == 0:
        bbox = (np.nan, np.nan, np.nan, np.nan)
        bbox_fraction = np.nan
        foreground_fraction = 0.0
    else:
        x0, x1 = int(xs.min()), int(xs.max()) + 1
        y0, y1 = int(ys.min()), int(ys.max()) + 1
        bbox = (x0, y0, x1, y1)
        bbox_fraction = ((x1 - x0) * (y1 - y0)) / (h * w)
        foreground_fraction = float(cleaned.mean())

    valid = (
        np.isfinite(bbox_fraction)
        and 0.005 <= foreground_fraction <= 0.95
        and 0.01 <= bbox_fraction <= 1.00
    )
    return cleaned, {
        'background_r': float(background_rgb[0]),
        'background_g': float(background_rgb[1]),
        'background_b': float(background_rgb[2]),
        'otsu_threshold_scaled': float(otsu_threshold),
        'border_width_px': int(border_width),
        'morphology_kernel_px': int(kernel_size),
        'bbox_x0': bbox[0], 'bbox_y0': bbox[1],
        'bbox_x1': bbox[2], 'bbox_y1': bbox[3],
        'bbox_area_fraction': float(bbox_fraction) if np.isfinite(bbox_fraction) else np.nan,
        'foreground_area_fraction': float(foreground_fraction),
        'extent_mask_valid': bool(valid),
    }


def peripheral_gradient_removal(rgb, zoom_factor):
    # Percentage of Sobel gradient energy outside the retained centre crop.
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float64)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.hypot(gx, gy)
    total = float(magnitude.sum())
    h, w = gray.shape

    crop_w = max(1, int(w / zoom_factor))
    crop_h = max(1, int(h / zoom_factor))
    left = (w - crop_w) // 2
    top = (h - crop_h) // 2
    retained = float(magnitude[top:top + crop_h, left:left + crop_w].sum())
    removed_pct = np.nan if total <= 0 else 100.0 * (total - retained) / total
    return {
        'gradient_energy_total': total,
        'gradient_energy_retained': retained,
        'gradient_removed_pct': removed_pct,
        'crop_x0': left, 'crop_y0': top,
        'crop_x1': left + crop_w, 'crop_y1': top + crop_h,
    }


def read_rgb(path):
    with Image.open(path) as image:
        return np.asarray(image.convert('RGB'))

In [ ]:
# 5. Extract object extent once per image and gradient removal at three zoom levels.
extent_rows = []
gradient_rows = []

for row in tqdm(images.itertuples(index=False), total=len(images), desc='Processing original images'):
    rgb = read_rgb(row.original_path)
    h, w = rgb.shape[:2]
    _, extent = clean_foreground_mask(rgb)
    extent_rows.append({
        'image_id': row.image_id,
        'class': row._0 if hasattr(row, '_0') else row[0],
        'original_filename': row.original_filename,
        'original_path': row.original_path,
        'width': w, 'height': h,
        **extent,
    })
    for level, factor in ZOOM_FACTORS.items():
        gradient_rows.append({
            'image_id': row.image_id,
            'class': row._0 if hasattr(row, '_0') else row[0],
            'original_filename': row.original_filename,
            'level': level,
            'zoom_factor': factor,
            **peripheral_gradient_removal(rgb, factor),
        })

extent_df = pd.DataFrame(extent_rows)
gradient_df = pd.DataFrame(gradient_rows)

# The namedtuple field for the column named "class" can vary by pandas version;
# overwrite it from the authoritative image table to guarantee correctness.
extent_df = extent_df.drop(columns=['class']).merge(
    images[['image_id', 'class']], on='image_id', how='left', validate='one_to_one'
)
gradient_df = gradient_df.drop(columns=['class']).merge(
    images[['image_id', 'class']], on='image_id', how='left', validate='many_to_one'
)

assert len(extent_df) == 2527
assert len(gradient_df) == 2527 * 3

extent_df.to_csv(TABLES / '01_object_extent_by_image.csv', index=False)
gradient_df.to_csv(TABLES / '02_peripheral_gradient_by_image_and_zoom.csv', index=False)

qc_summary = extent_df.groupby('class').agg(
    n=('image_id', 'size'),
    valid_n=('extent_mask_valid', 'sum'),
    valid_pct=('extent_mask_valid', lambda x: 100 * x.mean()),
    median_bbox_fraction=('bbox_area_fraction', 'median'),
    median_foreground_fraction=('foreground_area_fraction', 'median'),
).reset_index()
qc_summary.to_csv(TABLES / '03_object_extent_qc_summary.csv', index=False)

print('Valid extent masks:', int(extent_df.extent_mask_valid.sum()), '/', len(extent_df),
      f'({100 * extent_df.extent_mask_valid.mean():.2f}%)')
display(qc_summary)

In [ ]:
# 6. Visual quality-control montage for the thresholded foreground masks.
sampled = (images.groupby('class', group_keys=False)
           .sample(n=4, random_state=SEED)
           .reset_index(drop=True))

fig, axes = plt.subplots(6, 4, figsize=(12, 15), constrained_layout=True)
for ax, row in zip(axes.ravel(), sampled.itertuples(index=False)):
    rgb = read_rgb(row.original_path)
    mask, extent = clean_foreground_mask(rgb)
    overlay = rgb.astype(np.float32).copy()
    red = np.zeros_like(overlay); red[..., 0] = 255
    overlay[mask > 0] = 0.65 * overlay[mask > 0] + 0.35 * red[mask > 0]
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    x0, y0, x1, y1 = [extent[k] for k in ['bbox_x0', 'bbox_y0', 'bbox_x1', 'bbox_y1']]
    if np.isfinite(x0):
        overlay_pil = Image.fromarray(overlay)
        draw = ImageDraw.Draw(overlay_pil)
        draw.rectangle([x0, y0, x1 - 1, y1 - 1], outline=(0, 255, 0), width=4)
        overlay = np.asarray(overlay_pil)
    ax.imshow(overlay)
    ax.set_title(f'{row._0 if hasattr(row, "_0") else row[0]} | {row.original_filename}\n'
                 f'bbox={extent["bbox_area_fraction"]:.3f}', fontsize=8)
    ax.axis('off')

fig.suptitle('R5 foreground-mask quality control: red mask and green bounding box', fontsize=14, fontweight='bold')
fig.savefig(QC / 'Figure_R5_foreground_mask_QC.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# 7. Merge image features with the existing Block5 zoom sensitivity metrics.
analysis = (zoom_metrics.merge(
        gradient_df[['image_id', 'class', 'original_filename', 'level', 'zoom_factor',
                     'gradient_removed_pct']],
        on=['image_id', 'class', 'original_filename', 'level'],
        how='inner', validate='one_to_one')
    .merge(
        extent_df[['image_id', 'bbox_area_fraction', 'foreground_area_fraction',
                   'extent_mask_valid']],
        on='image_id', how='left', validate='many_to_one'))

analysis['angular_sensitivity'] = 1.0 - analysis['cosine_similarity']
assert len(analysis) == 2527 * 3
assert analysis['gradient_removed_pct'].notna().all()

analysis.to_csv(TABLES / '04_R5_merged_individual_data.csv', index=False)
print('Merged rows:', len(analysis))
display(analysis.head())

## Verifying the old result before adding the new controls

The previous manuscript reported cardboard correlations of approximately 0.396, 0.330, and 0.278 for angular sensitivity at low, medium, and high zoom. The next cell checks whether the recomputed gradient measure reproduces those values. A small numerical difference can occur through rounding; a large difference requires inspection before interpreting R5.

In [ ]:
# 8. Audit against the previously reported cardboard result.
expected_old_rho = {'low': 0.396, 'medium': 0.330, 'high': 0.278}
expected_old_median = {'low': 14.41, 'medium': 31.43, 'high': 50.79}
audit_rows = []
for level in LEVELS:
    d = analysis[(analysis['class'] == 'cardboard') & (analysis['level'] == level)]
    rho, p = spearmanr(d['gradient_removed_pct'], d['angular_sensitivity'])
    median = d['gradient_removed_pct'].median()
    audit_rows.append({
        'level': level,
        'recomputed_rho': rho,
        'previously_reported_rho': expected_old_rho[level],
        'absolute_rho_difference': abs(rho - expected_old_rho[level]),
        'recomputed_median_gradient_removed_pct': median,
        'previously_reported_median_pct': expected_old_median[level],
        'absolute_median_difference_percentage_points': abs(median - expected_old_median[level]),
        'p_value': p,
    })
audit = pd.DataFrame(audit_rows)
audit.to_csv(TABLES / '05_audit_previous_cardboard_results.csv', index=False)
display(audit.round(4))

if (audit['absolute_rho_difference'] > 0.02).any() or \
   (audit['absolute_median_difference_percentage_points'] > 1.0).any():
    warnings.warn(
        'The recomputed gradient analysis does not closely reproduce at least one previous value. '
        'Do not interpret the final R5 tables until this discrepancy is reviewed.'
    )
else:
    print('Previous cardboard result reproduced within the predefined audit tolerances.')

In [ ]:
# 9. Bootstrap utilities and statistical helpers.
def bootstrap_median_ci(values, n_boot=N_BOOT_MEDIAN, seed=SEED):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    local_rng = np.random.default_rng(seed)
    medians = np.empty(n_boot)
    for i in range(n_boot):
        medians[i] = np.median(local_rng.choice(x, size=len(x), replace=True))
    return tuple(np.quantile(medians, [0.025, 0.975]))


def bootstrap_median_difference_ci(x, y, n_boot=N_BOOT_MEDIAN, seed=SEED):
    x = np.asarray(x, dtype=float); x = x[np.isfinite(x)]
    y = np.asarray(y, dtype=float); y = y[np.isfinite(y)]
    local_rng = np.random.default_rng(seed)
    differences = np.empty(n_boot)
    for i in range(n_boot):
        differences[i] = (
            np.median(local_rng.choice(x, size=len(x), replace=True))
            - np.median(local_rng.choice(y, size=len(y), replace=True))
        )
    return tuple(np.quantile(differences, [0.025, 0.975]))


def residualize(values, covariates):
    y = rankdata(np.asarray(values, dtype=float), method='average')
    Xcols = [np.ones(len(y))]
    for covariate in covariates:
        Xcols.append(rankdata(np.asarray(covariate, dtype=float), method='average'))
    X = np.column_stack(Xcols)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return y - X @ beta


def partial_spearman(x, y, covariates):
    arrays = [np.asarray(x, dtype=float), np.asarray(y, dtype=float)]
    arrays += [np.asarray(c, dtype=float) for c in covariates]
    valid = np.logical_and.reduce([np.isfinite(a) for a in arrays])
    arrays = [a[valid] for a in arrays]
    rx = residualize(arrays[0], arrays[2:])
    ry = residualize(arrays[1], arrays[2:])
    r = pearsonr(rx, ry).statistic
    n = len(rx)
    degrees_freedom = n - len(covariates) - 2
    t_stat = r * math.sqrt(degrees_freedom / max(1e-15, 1 - r ** 2))
    p = 2 * t_dist.sf(abs(t_stat), degrees_freedom)
    return float(r), float(p), int(n)


def bootstrap_partial_ci(data, xcol, ycol, covariate_cols, n_boot=N_BOOT_PARTIAL, seed=SEED):
    local_rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        idx = local_rng.integers(0, len(data), size=len(data))
        sample = data.iloc[idx]
        try:
            r, _, _ = partial_spearman(
                sample[xcol], sample[ycol], [sample[c] for c in covariate_cols]
            )
            if np.isfinite(r):
                values.append(r)
        except Exception:
            pass
    return tuple(np.quantile(values, [0.025, 0.975])) if values else (np.nan, np.nan)

In [ ]:
# 10. Test whether cardboard has the highest median peripheral-gradient removal.
class_summary_rows = []
for level in LEVELS:
    for class_name in CLASSES:
        x = analysis[(analysis['level'] == level) & (analysis['class'] == class_name)][
            'gradient_removed_pct'
        ]
        lo, hi = bootstrap_median_ci(x, seed=SEED + LEVELS.index(level) * 20 + CLASSES.index(class_name))
        class_summary_rows.append({
            'level': level, 'class': class_name, 'n': len(x),
            'median_gradient_removed_pct': x.median(),
            'bootstrap_95_ci_low': lo, 'bootstrap_95_ci_high': hi,
            'median_bbox_area_fraction': analysis[(analysis.level == level) & (analysis['class'] == class_name)][
                'bbox_area_fraction'].median(),
        })
class_gradient_summary = pd.DataFrame(class_summary_rows)
class_gradient_summary.to_csv(TABLES / '06_class_gradient_and_extent_summary.csv', index=False)

omnibus_rows = []
pairwise_rows = []
for level in LEVELS:
    groups = [analysis[(analysis.level == level) & (analysis['class'] == c)]['gradient_removed_pct'] for c in CLASSES]
    H, p_kw = kruskal(*groups)
    n_total = sum(len(g) for g in groups)
    eta_squared_h = max(0.0, (H - len(groups) + 1) / (n_total - len(groups)))
    omnibus_rows.append({'level': level, 'H': H, 'df': len(groups) - 1, 'p': p_kw,
                          'eta_squared_h': eta_squared_h})

    cardboard = analysis[(analysis.level == level) & (analysis['class'] == 'cardboard')]['gradient_removed_pct']
    local_rows = []
    for other in [c for c in CLASSES if c != 'cardboard']:
        comparison = analysis[(analysis.level == level) & (analysis['class'] == other)]['gradient_removed_pct']
        U, p = mannwhitneyu(cardboard, comparison, alternative='two-sided')
        median_diff = cardboard.median() - comparison.median()
        ci_lo, ci_hi = bootstrap_median_difference_ci(
            cardboard, comparison,
            seed=SEED + LEVELS.index(level) * 100 + CLASSES.index(other)
        )
        local_rows.append({
            'level': level, 'comparison': f'cardboard vs {other}',
            'cardboard_median': cardboard.median(), 'other_median': comparison.median(),
            'median_difference_cardboard_minus_other': median_diff,
            'bootstrap_95_ci_low': ci_lo, 'bootstrap_95_ci_high': ci_hi,
            'U': U, 'p': p,
            'rank_biserial_cardboard_minus_other': 2 * U / (len(cardboard) * len(comparison)) - 1,
        })
    adjusted = multipletests([r['p'] for r in local_rows], method='holm')[1]
    for row, p_holm in zip(local_rows, adjusted):
        row['p_holm_within_level'] = p_holm
        pairwise_rows.append(row)

gradient_omnibus = pd.DataFrame(omnibus_rows)
gradient_omnibus['p_holm_across_levels'] = multipletests(gradient_omnibus['p'], method='holm')[1]
gradient_pairwise = pd.DataFrame(pairwise_rows)
gradient_omnibus.to_csv(TABLES / '07_gradient_class_omnibus_tests.csv', index=False)
gradient_pairwise.to_csv(TABLES / '08_cardboard_gradient_pairwise_tests.csv', index=False)

display(class_gradient_summary.round(3))
display(gradient_pairwise.round(4))

In [ ]:
# 11. Raw and object-extent-adjusted correlations for all six classes and both metrics.
OUTCOMES = {
    'Angular sensitivity (1 - cosine)': 'angular_sensitivity',
    'Relative Euclidean distance': 'relative_euclidean_distance',
}

correlation_rows = []
for metric_label, outcome in OUTCOMES.items():
    for level in LEVELS:
        for class_name in CLASSES:
            d_all = analysis[(analysis.level == level) & (analysis['class'] == class_name)].copy()
            raw_rho, raw_p = spearmanr(d_all['gradient_removed_pct'], d_all[outcome])

            d = d_all[d_all['extent_mask_valid']].dropna(
                subset=['gradient_removed_pct', outcome, 'bbox_area_fraction', 'foreground_area_fraction']
            )
            partial_bbox_rho, partial_bbox_p, n = partial_spearman(
                d['gradient_removed_pct'], d[outcome], [d['bbox_area_fraction']]
            )
            ci_lo, ci_hi = bootstrap_partial_ci(
                d, 'gradient_removed_pct', outcome, ['bbox_area_fraction'],
                seed=SEED + 1000 * list(OUTCOMES).index(metric_label)
                     + 100 * LEVELS.index(level) + CLASSES.index(class_name)
            )
            partial_area_rho, partial_area_p, _ = partial_spearman(
                d['gradient_removed_pct'], d[outcome], [d['foreground_area_fraction']]
            )
            correlation_rows.append({
                'metric': metric_label, 'level': level, 'class': class_name,
                'n_raw': len(d_all), 'n_adjusted': n,
                'raw_spearman_rho': raw_rho, 'raw_p': raw_p,
                'partial_spearman_bbox_rho': partial_bbox_rho,
                'partial_spearman_bbox_ci_low': ci_lo,
                'partial_spearman_bbox_ci_high': ci_hi,
                'partial_spearman_bbox_p': partial_bbox_p,
                'partial_spearman_foreground_area_rho': partial_area_rho,
                'partial_spearman_foreground_area_p': partial_area_p,
            })

correlations = pd.DataFrame(correlation_rows)
correlations['raw_p_holm'] = np.nan
correlations['partial_bbox_p_holm'] = np.nan
correlations['partial_foreground_p_holm'] = np.nan
for metric_label in OUTCOMES:
    for level in LEVELS:
        idx = correlations.index[(correlations.metric == metric_label) & (correlations.level == level)]
        correlations.loc[idx, 'raw_p_holm'] = multipletests(correlations.loc[idx, 'raw_p'], method='holm')[1]
        correlations.loc[idx, 'partial_bbox_p_holm'] = multipletests(
            correlations.loc[idx, 'partial_spearman_bbox_p'], method='holm')[1]
        correlations.loc[idx, 'partial_foreground_p_holm'] = multipletests(
            correlations.loc[idx, 'partial_spearman_foreground_area_p'], method='holm')[1]

correlations.to_csv(TABLES / '09_correlations_all_classes_both_metrics.csv', index=False)
display(correlations.round(4))

In [ ]:
# 12. Pooled rank-based models with class-by-gradient interaction and object-extent control.
def zscore_rank(series):
    ranked = rankdata(series, method='average')
    return (ranked - ranked.mean()) / ranked.std(ddof=1)


model_rows = []
slope_rows = []
valid_analysis = analysis[analysis['extent_mask_valid']].dropna(
    subset=['gradient_removed_pct', 'bbox_area_fraction']
).copy()

for metric_label, outcome in OUTCOMES.items():
    for level in LEVELS:
        d = valid_analysis[valid_analysis.level == level].copy()
        d['outcome_rank_z'] = zscore_rank(d[outcome])
        d['gradient_rank_z'] = zscore_rank(d['gradient_removed_pct'])
        d['extent_rank_z'] = zscore_rank(d['bbox_area_fraction'])
        d['class_name'] = d['class'].astype(str)

        formula = (
            'outcome_rank_z ~ gradient_rank_z * '
            'C(class_name, Treatment(reference="cardboard")) + extent_rank_z'
        )
        fit = smf.ols(formula, data=d).fit(cov_type='HC3')
        names = list(fit.params.index)
        interaction_names = [name for name in names
                             if 'gradient_rank_z:' in name or ':gradient_rank_z' in name]
        R = np.zeros((len(interaction_names), len(names)))
        for i, name in enumerate(interaction_names):
            R[i, names.index(name)] = 1
        wald = fit.wald_test(R, scalar=True)

        model_rows.append({
            'metric': metric_label, 'level': level, 'n': int(fit.nobs),
            'r_squared': fit.rsquared, 'adjusted_r_squared': fit.rsquared_adj,
            'class_by_gradient_wald_statistic': float(wald.statistic),
            'class_by_gradient_df': len(interaction_names),
            'class_by_gradient_p': float(wald.pvalue),
            'object_extent_beta': fit.params['extent_rank_z'],
            'object_extent_robust_se': fit.bse['extent_rank_z'],
            'object_extent_p': fit.pvalues['extent_rank_z'],
        })

        covariance = fit.cov_params().to_numpy()
        parameters = fit.params.to_numpy()
        for class_name in CLASSES:
            contrast = np.zeros(len(names))
            contrast[names.index('gradient_rank_z')] = 1
            if class_name != 'cardboard':
                candidate = [name for name in interaction_names if f'[T.{class_name}]' in name]
                if len(candidate) != 1:
                    raise RuntimeError(f'Could not identify interaction coefficient for {class_name}: {candidate}')
                contrast[names.index(candidate[0])] = 1
            estimate = float(contrast @ parameters)
            se = float(np.sqrt(contrast @ covariance @ contrast))
            z = estimate / se
            p = 2 * scipy.stats.norm.sf(abs(z))
            slope_rows.append({
                'metric': metric_label, 'level': level, 'class': class_name,
                'adjusted_gradient_slope': estimate,
                'robust_se': se,
                'ci_low': estimate - 1.96 * se,
                'ci_high': estimate + 1.96 * se,
                'p': p,
            })

pooled_models = pd.DataFrame(model_rows)
pooled_models['class_by_gradient_p_holm'] = multipletests(
    pooled_models['class_by_gradient_p'], method='holm')[1]
pooled_models['object_extent_p_holm'] = multipletests(
    pooled_models['object_extent_p'], method='holm')[1]
pooled_slopes = pd.DataFrame(slope_rows)

pooled_models.to_csv(TABLES / '10_pooled_class_interaction_models.csv', index=False)
pooled_slopes.to_csv(TABLES / '11_adjusted_class_specific_slopes.csv', index=False)
display(pooled_models.round(4))
display(pooled_slopes.round(4))

In [ ]:
# 13. Publication-quality figures.
sns.set_theme(style='whitegrid', context='paper')
class_order = CLASSES
palette = ['#4C78A8', '#72B7B2', '#F58518', '#E45756', '#B279A2', '#9D755D']

# Figure 1: gradient removal by class and zoom level.
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True, constrained_layout=True)
for ax, level in zip(axes, LEVELS):
    d = analysis[analysis.level == level]
    sns.boxplot(data=d, x='class', y='gradient_removed_pct', order=class_order,
                color='#7FA6C9', showfliers=False, ax=ax)
    ax.set_title(f'({chr(97 + LEVELS.index(level))}) {level.capitalize()} zoom ({ZOOM_FACTORS[level]:.2f}×)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)
    ax.set_ylabel('Peripheral gradient removed (%)' if level == 'low' else '')
fig.suptitle('Peripheral-gradient removal across TrashNet classes', fontsize=13, fontweight='bold')
fig.savefig(FIGURES / 'Figure_R5_gradient_removal_all_classes.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURES / 'Figure_R5_gradient_removal_all_classes.pdf', bbox_inches='tight')
plt.show()

# Figure 2: partial correlations controlling for bounding-box fraction.
fig, axes = plt.subplots(1, 2, figsize=(10.5, 5), constrained_layout=True)
for ax, metric_label in zip(axes, OUTCOMES):
    matrix = (correlations[correlations.metric == metric_label]
              .pivot(index='class', columns='level', values='partial_spearman_bbox_rho')
              .loc[class_order, LEVELS])
    sns.heatmap(matrix, annot=True, fmt='.2f', cmap='vlag', center=0,
                vmin=-0.6, vmax=0.6, cbar_kws={'label': 'Partial Spearman ρ'}, ax=ax)
    ax.set_title(metric_label)
    ax.set_xlabel('Zoom level')
    ax.set_ylabel('Class' if ax is axes[0] else '')
fig.suptitle('Gradient–sensitivity association controlling for object extent', fontsize=13, fontweight='bold')
fig.savefig(FIGURES / 'Figure_R5_partial_correlations_all_classes.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURES / 'Figure_R5_partial_correlations_all_classes.pdf', bbox_inches='tight')
plt.show()

# Figure 3: object-extent distribution by class.
fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
sns.boxplot(data=extent_df[extent_df.extent_mask_valid], x='class', y='bbox_area_fraction',
            order=class_order, palette=palette, showfliers=False, ax=ax)
ax.set_xlabel('TrashNet class')
ax.set_ylabel('Foreground bounding-box fraction')
ax.set_title('Estimated object extent used as the R5 covariate', fontweight='bold')
ax.tick_params(axis='x', rotation=25)
fig.savefig(FIGURES / 'Figure_R5_object_extent_by_class.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURES / 'Figure_R5_object_extent_by_class.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# 14. Generate a concise machine-readable result summary and environment record.
cardboard_partial = correlations[correlations['class'] == 'cardboard'][[
    'metric', 'level', 'n_adjusted', 'raw_spearman_rho',
    'partial_spearman_bbox_rho', 'partial_spearman_bbox_ci_low',
    'partial_spearman_bbox_ci_high', 'partial_bbox_p_holm'
]].copy()
cardboard_partial.to_csv(TABLES / '12_cardboard_raw_and_adjusted_correlations.csv', index=False)

highest_checks = []
for level in LEVELS:
    s = class_gradient_summary[class_gradient_summary.level == level].sort_values(
        'median_gradient_removed_pct', ascending=False)
    comparisons = gradient_pairwise[gradient_pairwise.level == level]
    highest_checks.append({
        'level': level,
        'class_with_highest_observed_median': s.iloc[0]['class'],
        'highest_observed_median_pct': s.iloc[0]['median_gradient_removed_pct'],
        'cardboard_highest_observed': bool(s.iloc[0]['class'] == 'cardboard'),
        'cardboard_significantly_higher_than_all_five_after_holm': bool(
            ((comparisons['median_difference_cardboard_minus_other'] > 0)
             & (comparisons['p_holm_within_level'] < 0.05)).all()
        ),
    })
highest_checks = pd.DataFrame(highest_checks)
highest_checks.to_csv(TABLES / '13_cardboard_highest_median_claim_check.csv', index=False)

validity = {
    'input_metrics_csv': str(METRICS_CSV),
    'dataset_root': str(DATASET_ROOT),
    'original_images': int(len(images)),
    'merged_analysis_rows': int(len(analysis)),
    'valid_object_extent_n': int(extent_df.extent_mask_valid.sum()),
    'valid_object_extent_pct': float(100 * extent_df.extent_mask_valid.mean()),
    'previous_result_rho_max_abs_difference': float(audit.absolute_rho_difference.max()),
    'previous_result_median_max_abs_difference_pp': float(
        audit.absolute_median_difference_percentage_points.max()),
    'versions': VERSIONS,
}
with open(OUTPUT_ROOT / 'entorno_y_validacion_R5.json', 'w', encoding='utf-8') as f:
    json.dump(validity, f, indent=2, ensure_ascii=False)

display(highest_checks)
display(cardboard_partial.round(4))

## How the decreasing correlations must be interpreted

Do **not** write that the decrease in ρ proves that peripheral-gradient removal “accounts for part, but not all” of the alteration. The revised interpretation should be based on the adjusted results:

> The strength of the association varied across zoom levels. Differences between coefficients may reflect changes in the range of removed gradient energy, nonlinear responses, object-extent confounding, and the additional rescaling component of center-crop zoom. The correlations identify association and do not quantify a causal proportion of representational change.

The final wording will be written after reviewing the output ZIP.

In [ ]:
# 15. Package every R5 result into one ZIP for download and review.
README = OUTPUT_ROOT / 'LEEME_R5.txt'
README.write_text(
    '''R5 — Peripheral gradient and object extent\n\n'''
    'This package was generated from the 2,527 original TrashNet images and the existing '
    'complete Block5 zoom metrics. It contains image-level object-extent estimates, '
    'recomputed peripheral-gradient removal, all-class raw and partial correlations, '
    'cardboard median tests, pooled class-interaction models, quality-control material, '
    'and publication-quality figures.\n\n'
    'Upload the complete ZIP to ChatGPT/Codex for interpretation and manuscript revision.\n',
    encoding='utf-8'
)

zip_path = OUTPUT_ROOT.parent / 'Revision_R5_Peripheral_Gradient_VGG16.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_ROOT.rglob('*')):
        if path.is_file():
            zf.write(path, arcname=str(Path(OUTPUT_ROOT.name) / path.relative_to(OUTPUT_ROOT)))

print('R5 package created successfully:')
print(zip_path)
print('Size (MB):', round(zip_path.stat().st_size / 1024 ** 2, 2))

from google.colab import files
files.download(str(zip_path))